# CKP02 — DocMind RAG
## FinControl Assistant

### Retrieval-Augmented Generation para Educação Financeira Pessoal

**Equipe:** CKP02_FinControlAssistant_Equipe1

Este notebook apresenta o desenvolvimento e a avaliação de um pipeline
RAG aplicado ao domínio de educação financeira pessoal.

O pipeline implementado segue as etapas:

**Load → Split → Embed → Store → Retrieve → Generate**

São comparadas duas estratégias de chunking:

- Chunk de 512 caracteres, com overlap de 51 caracteres;
- Chunk de 1024 caracteres, com overlap de 102 caracteres.

A avaliação quantitativa utilizando RAGAS foi realizada separadamente,
considerando as métricas **Faithfulness** e **Answer Relevancy**, e implementada ao final do notebook.

## 1. Objetivo

O objetivo deste projeto é construir um sistema RAG capaz de responder
perguntas relacionadas à organização e ao planejamento financeiro pessoal
utilizando exclusivamente informações presentes em uma base documental
composta por fontes oficiais.

O sistema utiliza:

- documentos reais de instituições oficiais;
- `RecursiveCharacterTextSplitter`;
- embeddings `nomic-embed-text`;
- ChromaDB como banco vetorial;
- `gemma4:cloud` como modelo de geração;
- RAGAS para avaliação da qualidade das respostas.

Além disso, são comparadas duas configurações de chunking para analisar
o impacto do tamanho dos chunks na recuperação e na geração das respostas.

In [1]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Diretório atual:", Path.cwd())
print("Projeto:", PROJECT_ROOT)

Diretório atual: c:\Users\lucas_w312nz\Downloads\CKP02_FinControlAssistant_Equipe1
Projeto: c:\Users\lucas_w312nz\Downloads\CKP02_FinControlAssistant_Equipe1


In [2]:
import subprocess

resultado = subprocess.run(
    ["ollama", "--version"],
    capture_output=True,
    text=True
)

print(resultado.stdout.strip())

ollama version is 0.40.1


In [3]:
resultado = subprocess.run(
    ["ollama", "list"],
    capture_output=True,
    text=True
)

print(resultado.stdout)

NAME                       ID              SIZE      MODIFIED   
nomic-embed-text:latest    0a109f422b47    274 MB    2 days ago    



## 2. Base documental

A base de conhecimento é composta por documentos relacionados à educação
financeira e ao planejamento financeiro pessoal.

| ID | Instituição | Documento | Tema | Fonte oficial |
|---|---|---|---|---|
| DOC01 | Banco Central do Brasil | Caderno de Educação Financeira | Educação financeira | Banco Central |
| DOC02 | CVM | TOP Planejamento Financeiro Pessoal | Planejamento financeiro | Portal do Investidor |
| DOC03 | CVM | Guia de Planejamento Financeiro | Planejamento financeiro | Portal do Investidor |
| DOC04 | CAIXA | Receitas e Despesas | Receitas e despesas | CAIXA |
| DOC05 | SUSEP | Orçamento | Orçamento pessoal | Gov.br |

Os documentos foram selecionados a partir de fontes institucionais
oficiais.

In [4]:
from pathlib import Path

DATA_DIR = PROJECT_ROOT / "data"

documentos = sorted(DATA_DIR.glob("*.pdf"))

for documento in documentos:
    print(documento.name)

print(f"\nTotal de documentos: {len(documentos)}")

DOC01_BCB_Caderno_Educacao_Financeira.pdf
DOC02_CVM_TOP_Planejamento_Financeiro_Pessoal.pdf
DOC03_CVM_Guia_Planejamento_Financeiro.pdf
DOC04_CAIXA_Receitas_e_Despesas.pdf
DOC05_SUSEP_Orcamento.pdf

Total de documentos: 5


## 3. Estratégias de chunking

Para analisar o impacto do tamanho dos chunks na recuperação semântica,
foram utilizadas duas configurações.

### Configuração 1

- Chunk size: 512
- Overlap: 51
- Proporção de overlap: aproximadamente 10%

### Configuração 2

- Chunk size: 1024
- Overlap: 102
- Proporção de overlap: aproximadamente 10%

Ambas utilizam o `RecursiveCharacterTextSplitter` com os separadores:

`["\n\n", "\n", ". ", " ", ""]`.

In [5]:
from app.vector_store import processar_e_armazenar_documentos

processar_e_armazenar_documentos()

Carregado: data/DOC01_BCB_Caderno_Educacao_Financeira.pdf
Carregado: data/DOC02_CVM_TOP_Planejamento_Financeiro_Pessoal.pdf
Carregado: data/DOC03_CVM_Guia_Planejamento_Financeiro.pdf
Carregado: data/DOC04_CAIXA_Receitas_e_Despesas.pdf
Carregado: data/DOC05_SUSEP_Orcamento.pdf
Iniciando embeddings via Ollama (nomic-embed-text)...
Armazenando chunks de 512 na coleção 'fin_control_512'...
Armazenando chunks de 1024 na coleção 'fin_control_1024'...
Sistemas prontos! 72 chunks (512) e 40 chunks (1024) foram armazenados com sucesso nas suas respectivas coleções.


## 4. Embeddings e armazenamento vetorial

Os documentos são transformados em representações vetoriais utilizando
o modelo:

`nomic-embed-text`

Os vetores são armazenados localmente no ChromaDB em duas coleções
independentes:

- `fin_control_512`
- `fin_control_1024`

A separação das coleções permite comparar as duas estratégias de chunking
utilizando a mesma base documental e o mesmo modelo de embedding.

In [6]:
import chromadb

client = chromadb.PersistentClient(path="./chroma_db")

colecoes = client.list_collections()

for colecao in colecoes:
    print(
        colecao.name,
        "→",
        colecao.count(),
        "chunks"
    )

fin_control_512 → 72 chunks
fin_control_1024 → 40 chunks


In [7]:
colecao_512 = client.get_collection("fin_control_512")
colecao_1024 = client.get_collection("fin_control_1024")

assert colecao_512.count() == 72, (
    f"Esperados 72 chunks, encontrados {colecao_512.count()}"
)

assert colecao_1024.count() == 40, (
    f"Esperados 40 chunks, encontrados {colecao_1024.count()}"
)

print("✓ fin_control_512: 72 chunks")
print("✓ fin_control_1024: 40 chunks")
print("✓ Coleções validadas com sucesso.")

✓ fin_control_512: 72 chunks
✓ fin_control_1024: 40 chunks
✓ Coleções validadas com sucesso.


In [8]:
from app.vector_store import buscar

pergunta = "O que é um orçamento pessoal?"

resultados = buscar(
    pergunta,
    tamanho_chunk=512,
    k=3
)

for documento, similaridade in resultados:
    print("=" * 70)
    print("Arquivo:", documento.metadata.get("source"))
    print("Página:", documento.metadata.get("page"))
    print("Similaridade:", round(similaridade, 4))
    print(documento.page_content[:500])

Arquivo: data/DOC05_SUSEP_Orcamento.pdf
Página: 0
Similaridade: 0.7571
DOC05 — Orçamento (SUSEP / Gov.br)
Categoria: Orçamento pessoal | Uso no FinControl Assistant: orientações sobre elaboração e controle do
orçamento pessoal
Fonte: https://www.gov.br/susep/pt-br/assuntos/meu-futuro-seguro/educacao-financeira/orcamento-1
(publicado em 15/09/2022, modificado em 05/10/2022). Resumo organizado, não é cópia literal.
1. O que é o orçamento pessoal ou familiar
É o registro das receitas e despesas, previstas e realizadas, de uma pessoa ou família.
Arquivo: data/DOC01_BCB_Caderno_Educacao_Financeira.pdf
Página: 1
Similaridade: 0.7339
particulares); despesas fixas (aluguel, prestação) e variáveis (luz, água, conforme o consumo).
Situações do orçamento: deficitário (despesas maiores que receitas), neutro (iguais) e superavitário
(menores). Meta básica: manter o orçamento superavitário.
Pague-se primeiro: ao receber a renda, separar a parte da poupança antes de pagar as despesas, de
preferência 

## 5. Recuperação semântica

In [9]:
from app.vector_store import buscar

pergunta = "O que é um orçamento pessoal?"

for tamanho in [512, 1024]:

    print("\n" + "=" * 70)
    print(f"RETRIEVAL — CHUNK {tamanho}")
    print("=" * 70)

    resultados = buscar(
        pergunta,
        tamanho_chunk=tamanho,
        k=3
    )

    for i, (documento, similaridade) in enumerate(resultados, start=1):
        print(f"\nResultado {i}")
        print(f"Arquivo: {documento.metadata.get('source')}")
        print(f"Página: {documento.metadata.get('page')}")
        print(f"Similaridade: {similaridade:.4f}")
        print(f"Trecho: {documento.page_content[:500]}...")


RETRIEVAL — CHUNK 512

Resultado 1
Arquivo: data/DOC05_SUSEP_Orcamento.pdf
Página: 0
Similaridade: 0.7571
Trecho: DOC05 — Orçamento (SUSEP / Gov.br)
Categoria: Orçamento pessoal | Uso no FinControl Assistant: orientações sobre elaboração e controle do
orçamento pessoal
Fonte: https://www.gov.br/susep/pt-br/assuntos/meu-futuro-seguro/educacao-financeira/orcamento-1
(publicado em 15/09/2022, modificado em 05/10/2022). Resumo organizado, não é cópia literal.
1. O que é o orçamento pessoal ou familiar
É o registro das receitas e despesas, previstas e realizadas, de uma pessoa ou família....

Resultado 2
Arquivo: data/DOC01_BCB_Caderno_Educacao_Financeira.pdf
Página: 1
Similaridade: 0.7339
Trecho: particulares); despesas fixas (aluguel, prestação) e variáveis (luz, água, conforme o consumo).
Situações do orçamento: deficitário (despesas maiores que receitas), neutro (iguais) e superavitário
(menores). Meta básica: manter o orçamento superavitário.
Pague-se primeiro: ao receber a renda, sep

## 6. Geração das respostas

Após a recuperação dos chunks relevantes, o contexto recuperado é enviado
ao modelo `gemma4:cloud`.

O modelo recebe instruções para:

1. utilizar somente as informações recuperadas;
2. não inventar informações;
3. informar quando a resposta não estiver presente na base;
4. apresentar referências aos documentos utilizados.

In [10]:
from app.rag import responder

resultado = responder(
    "O que é um orçamento pessoal?",
    tamanho_chunk=512,
    k=3
)

print(resultado["resposta"])

print("\nFontes:")
for fonte in resultado["fontes"]:
    print(f"- {fonte}")

O orçamento pessoal ou familiar é o registro das receitas e despesas, tanto as previstas quanto as realizadas, de uma pessoa ou família [1].

Fontes:
- {'indice': 1, 'arquivo': 'DOC05_SUSEP_Orcamento.pdf', 'pagina': 1, 'similaridade': 0.7571, 'trecho': 'DOC05 — Orçamento (SUSEP / Gov.br) Categoria: Orçamento pessoal | Uso no FinControl Assistant: orientações sobre elaboração e controle do orçamento pessoal Fonte: https://www.gov.br/susep/pt-br/assuntos/meu-futuro-seguro/educacao-financeira/orcamento-1 (publicado em 15/09/2022, modificado em 05/10/2...'}
- {'indice': 2, 'arquivo': 'DOC01_BCB_Caderno_Educacao_Financeira.pdf', 'pagina': 2, 'similaridade': 0.7339, 'trecho': 'particulares); despesas fixas (aluguel, prestação) e variáveis (luz, água, conforme o consumo). Situações do orçamento: deficitário (despesas maiores que receitas), neutro (iguais) e superavitário (menores). Meta básica: manter o orçamento superavitário. Pague-se primeiro: ao receber a renda, separa...'}
- {'indice': 3

## 7. Pipeline RAG completo

O pipeline desenvolvido integra todas as etapas:

**Load**
→ leitura dos documentos PDF

**Split**
→ divisão utilizando duas configurações de chunking

**Embed**
→ `nomic-embed-text`

**Store**
→ ChromaDB

**Retrieve**
→ busca semântica dos chunks mais relevantes

**Generate**
→ `gemma4:cloud`

In [11]:
pergunta = "Como organizar as receitas e despesas pessoais?"

for tamanho in [512, 1024]:
    print("\n" + "=" * 80)
    print(f"CONFIGURAÇÃO: CHUNK {tamanho}")
    print("=" * 80)

    resultado = responder(
        pergunta,
        tamanho_chunk=tamanho,
        k=3
    )

    print(resultado["resposta"])

    print("\nFontes:")
    for fonte in resultado["fontes"]:
        print("-", fonte)


CONFIGURAÇÃO: CHUNK 512
Para organizar as receitas e despesas, a regra básica é que as despesas não superem as receitas, sendo prudente que as receitas sejam maiores para formar poupança [3]. O processo pode ser feito em etapas, começando pelo planejamento para estimar valores com base na rotina passada, separando gastos fixos, variáveis, sazonais e compromissos assumidos [3]. Recomenda-se registrar receitas e despesas por pelo menos um mês em grupos como habitação, alimentação, saúde, transporte, educação e lazer [2]. É importante considerar rendas extras anuais, como 13º salário e férias, além de despesas não mensais, como seguros e impostos [2]. Uma sugestão é subdividir os gastos em essenciais, não essenciais, investimentos e financiamentos [2].

Fontes:
- {'indice': 1, 'arquivo': 'DOC04_CAIXA_Receitas_e_Despesas.pdf', 'pagina': 1, 'similaridade': 0.7597, 'trecho': 'receitas maiores que as despesas. A página reconhece que o conceito parece óbvio, mas nem sempre é fácil de aplicar 

## 8. Validação funcional

Além das consultas relacionadas ao domínio da base, é realizada uma
consulta fora do escopo documental para verificar o comportamento de
fallback do sistema.

In [12]:
resultado = responder(
    "Qual é a capital da Austrália?",
    tamanho_chunk=512,
    k=3
)

print("Resposta:")
print(resultado["resposta"])

Resposta:
Não encontrei essa informação nos documentos da base.


## 8. Resultados da avaliação RAGAS

A avaliação foi realizada com seis perguntas sobre educação financeira, utilizando duas configurações de chunking: 512 e 1024 caracteres, com recuperação de `k=3` trechos por consulta. Foram avaliadas as métricas **Faithfulness** e **Answer Relevancy** para cada pergunta e configuração.

### 8.1. Métricas por pergunta

| ID        | Categoria               | Faithfulness (512) | Answer Relevancy (512) | Faithfulness (1024) | Answer Relevancy (1024) |
| --------- | ----------------------- | -----------------: | ---------------------: | ------------------: | ----------------------: |
| P1        | Orçamento               |             0,8333 |                 0,7386 |              1,0000 |                  0,8666 |
| P2        | Receitas e despesas     |             1,0000 |                 0,9534 |              1,0000 |                  0,8895 |
| P3        | Educação financeira     |             1,0000 |                 0,9860 |              1,0000 |                  0,9318 |
| P4        | Planejamento financeiro |             1,0000 |                 0,8758 |              1,0000 |                  0,7966 |
| P5        | Reserva financeira      |             1,0000 |                 0,7832 |              1,0000 |                  0,5804 |
| P6        | Receitas e despesas     |             1,0000 |                 0,6523 |              1,0000 |                  0,0000 |
| **Média** | **Todas as categorias** |         **0,9722** |             **0,8315** |          **1,0000** |              **0,6775** |

### 8.2. Perguntas avaliadas e recuperação das fontes

| ID        | Pergunta                                                           | Fonte esperada recuperada (512) | Fonte esperada recuperada (1024) |
| --------- | ------------------------------------------------------------------ | ------------------------------- | -------------------------------- |
| P1        | O que é um orçamento pessoal e para que ele serve?                 | Sim                             | Sim                              |
| P2        | Qual a diferença entre despesas fixas e variáveis?                 | Não                             | Sim                              |
| P3        | Por que é importante ter educação financeira?                      | Sim                             | Sim                              |
| P4        | Quais são as etapas para fazer um planejamento financeiro pessoal? | Sim                             | Sim                              |
| P5        | Quanto dinheiro devo ter separado para imprevistos?                | Sim                             | Sim                              |
| P6        | O que fazer quando as despesas são maiores que a receita?          | Sim                             | Sim                              |
| **Total** | **6 perguntas**                                                    | **5/6 (83,33%)**                | **6/6 (100%)**                   |

### 8.3. Configurações e contexto recuperado

| Configuração   | Tamanho do chunk |        Overlap | Chunks recuperados (`k`) | Tamanho médio do contexto |
| -------------- | ---------------: | -------------: | -----------------------: | ------------------------: |
| Configuração A |   512 caracteres |  51 caracteres |                        3 |            399 caracteres |
| Configuração B |  1024 caracteres | 102 caracteres |                        3 |            900 caracteres |

**Observação:** o overlap foi configurado em aproximadamente 10% do tamanho de cada chunk. A recuperação da fonte esperada indica se ao menos um dos documentos esperados para aquela pergunta apareceu entre os documentos recuperados; isso não substitui a avaliação de relevância semântica realizada pelo RAGAS.


## Análise crítica dos resultados e escolha da configuração

### Comparação das métricas

A avaliação foi realizada com seis perguntas, utilizando três chunks recuperados por consulta (`k=3`) e comparando duas configurações de chunking: 512 e 1024 caracteres.

Os resultados médios obtidos foram:

| Métrica                       |      Chunk 512 |     Chunk 1024 |
| ----------------------------- | -------------: | -------------: |
| Faithfulness                  |         0,9722 |         1,0000 |
| Answer relevancy              |         0,8315 |         0,6775 |
| Recuperação da fonte esperada |         83,33% |           100% |
| Tamanho médio do contexto     | 399 caracteres | 900 caracteres |

Ambas as configurações superaram o requisito mínimo de faithfulness de 0,70 e atingiram a referência de 0,90. Entretanto, os resultados mostram diferenças entre as configurações: o chunk 1024 apresentou maior faithfulness média, enquanto o chunk 512 obteve maior answer relevancy média.

### Análise crítica

A configuração com chunks de 1024 apresentou faithfulness médio de 1,00 e recuperou a fonte esperada nas seis perguntas avaliadas. Apesar disso, sua answer relevancy média foi inferior à da configuração de 512 caracteres.

Um caso relevante ocorreu na pergunta P6: “O que fazer quando as despesas são maiores que a receita?”. Com chunk 1024, a resposta explicou o significado de um resultado deficitário, mas não apresentou ações específicas para reverter a situação. Os contextos recuperados nessa configuração também não continham as orientações específicas que estavam disponíveis na recuperação com chunk 512. A answer relevancy dessa resposta foi avaliada em 0,00.

Esse caso demonstra que uma pontuação elevada de faithfulness, isoladamente, não garante que a resposta atenda integralmente à pergunta. A fidelidade ao contexto e a relevância da resposta devem ser analisadas em conjunto.

A recuperação da fonte esperada também deve ser interpretada como um indicador auxiliar: não encontrar o documento previamente esperado não significa necessariamente que os trechos recuperados sejam irrelevantes ou incorretos.

### Escolha da configuração

Com base nos resultados, adotamos provisoriamente chunks de 512 caracteres como configuração padrão do projeto. Essa escolha considera o equilíbrio entre faithfulness elevada (0,9722) e maior answer relevancy média (0,8315).

A configuração de 1024 caracteres continua sendo uma alternativa válida, especialmente porque apresentou faithfulness de 1,00 e recuperou a fonte esperada em todas as perguntas avaliadas. No entanto, sua menor relevância média indica que fornecer mais contexto não garante, por si só, respostas mais adequadas à pergunta.

A escolha de 512 caracteres não representa uma superioridade em todas as métricas, mas uma decisão baseada no conjunto de resultados observados. Essa conclusão é provisória, pois a avaliação utilizou seis perguntas e o mesmo modelo, `gemma4:cloud`, tanto para gerar as respostas quanto para avaliar as métricas RAGAS.

### Limitações da avaliação

* Foram avaliadas seis perguntas, o que limita a generalização dos resultados para outras consultas.
* A recuperação utilizou `k=3`, isto é, até três trechos por pergunta.
* O mesmo modelo foi utilizado na geração das respostas e no julgamento RAGAS, o que pode influenciar as pontuações.
* A resposta P6 com chunk 1024 merece investigação adicional devido à baixa answer relevancy.

Portanto, os resultados fornecem evidências iniciais para a escolha da configuração, mas avaliações adicionais serão necessárias para confirmar seu desempenho em um conjunto maior e mais diversificado de perguntas.


## Conclusão

O projeto implementou um pipeline RAG completo sobre documentos reais de educação financeira, contemplando carregamento, divisão em chunks, embeddings, armazenamento no ChromaDB, recuperação de contexto e geração de respostas com citações de fontes.

A comparação entre chunks de 512 e 1024 caracteres demonstrou que ambas as configurações atingiram a meta de faithfulness estabelecida. A configuração de 512 caracteres apresentou melhor answer relevancy média, enquanto a de 1024 caracteres apresentou maior faithfulness e maior taxa de recuperação da fonte esperada.

Com base no conjunto de métricas, adotamos provisoriamente chunks de 512 caracteres como padrão, mantendo a configuração de 1024 para comparação e possíveis usos futuros. Essa decisão considera tanto a fidelidade ao contexto quanto a capacidade de responder de forma relevante à pergunta do usuário.

A avaliação RAGAS complementou os testes funcionais do pipeline, permitindo identificar limitações que não seriam evidentes apenas pela inspeção de algumas respostas. Como próximos passos, recomenda-se ampliar o conjunto de perguntas e investigar os casos de baixa relevância para aprimorar a recuperação de contexto.
